# PyCaret 4.0.0a3 — Colab smoke test

Verifies that `pycaret==4.0.0a3` installs and runs a full classification
loop on Google Colab. Upload this `.ipynb` to Colab and click **Runtime → Run all**.

If Colab prompts you to restart the runtime after the install cell, do so,
then re-run from the import cell down.

In [ ]:
# 1. Install PyCaret 4.0 alpha from PyPI.
# `--pre` is required because 4.0.0a3 is a PEP 440 pre-release.
!pip install --pre pycaret==4.0.0a3 -q

In [ ]:
# 2. Confirm the version that got installed.
import pycaret
print("PyCaret", pycaret.__version__)

In [ ]:
# 3. Load a built-in dataset and fit a ClassificationExperiment.
from pycaret.datasets import get_data
from pycaret.tasks import ClassificationExperiment

data = get_data("juice")
exp = ClassificationExperiment(
    target="Purchase",
    session_id=42,
    n_jobs=1,
).fit(data)

print("is_fitted:", exp.__sklearn_is_fitted__())
print("train:", exp.X_train.shape, "test:", exp.X_test.shape)

In [ ]:
# 4. Compare a few models. Returns a typed CompareResult.
result = exp.compare_models(include=["lr", "dt", "rf"])
result.leaderboard

In [ ]:
# 5. Tune the best model and predict on the held-out test set.
tuned = exp.tune_model(result.best, n_iter=5).pipeline
preds = exp.predict_model(tuned)
preds.predictions.head()

In [ ]:
# 6. Save and reload to confirm artifact round-trip.
from pycaret import save_model, load_model

out = save_model(tuned, "juice_classifier")
print("saved to:", out)

restored = load_model(out)
print("restored:", type(restored).__name__)

If every cell ran without error and step 6 prints `restored: LogisticRegression`
(or whatever model `compare_models` ranked first), the install is healthy.